In [2]:
from dotenv import load_dotenv
import os

# .envファイルの絶対パスを指定
env_path = os.path.join("..", ".env")  # 親ディレクトリの.envを指定
load_dotenv(env_path)

True

In [3]:
base_url = 'http://127.0.0.1:7070'
base_url

'http://127.0.0.1:7070'

In [72]:
from enum import Enum


class FurnitureType(Enum):
    TABLE="TABLE"
    TV="TV"
    

In [76]:
import httpx
import json

data = {
    "isInFov" : True, 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url+"/furniture/fov", json = data)
data = json.load(res)
data


{'status': 'success',
 'furnitures': ['{"position":{"x":-0.4238522,"y":-0.667920232,"z":4.70446253},"distance_from_user":4.770507,"id":"9e02c01c-98c3-4cad-ac61-098f36d02e0d","name":"テーブル","FurnitureType":"TABLE"}']}

In [ ]:
data = {
    "direction" : "FRONT", 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}


res = httpx.post(base_url + "/furniture/direction", json=data)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
furnitures = [json.loads(f) for f in data["furnitures"]]  # JSON文字列 → dictに変換

print(data)
print()
print(furnitures[0]["id"])


In [ ]:

sending = {
    "direction" : "FRONT", 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url + "/furniture/direction", json=sending)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
furnitures = [json.loads(f) for f in data["furnitures"]] # JSON文字列 → dictに変換

print(f"RECEIVED FURNITUR: name {furnitures[0]['name']}, id {furnitures[0]['id']}")



sending = {
    "id" : furnitures[0]['id'], 
    "order": "proximity", 
    "range": 0
}

print("sending", sending)
res = httpx.post(base_url + "/furniture/find_device_with_furniture", json=sending)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
data

In [109]:
lower("table")

NameError: name 'lower' is not defined

In [156]:
sending = {
   
    "range": 5, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url + "/furniture/get", json=sending)
data = res.json()
data["devices"] = [{"id": i["id"], "name" : i["name"], "position_relative_to_furniture" : i["position"]} for i in data["devices"]]
data["furniture_data"] = {
"furnitureShape": data["furniture_data"]["furnitureShape"],
"distance_from_user" : data["furniture_data"]["distance_from_user"],
"name" : data["furniture_data"]["name"] ,
    "id":  data["furniture_data"]["id"]


}
data

{'status': 'success',
 'devices': [{'id': '7394d028-d0fe-43c3-beea-b26816156615',
   'name': 'フロアライト',
   'position_relative_to_furniture': {'x': 0.725365639,
    'y': 0.768305957,
    'z': -0.06326866}},
  {'id': 'e0497b7d-38b5-4f78-86bc-dc6a53faaa5a',
   'name': 'フロアライト (1)',
   'position_relative_to_furniture': {'x': -0.3553723,
    'y': 0.7668657,
    'z': -0.02977848}}],
 'furniture_data': {'furnitureShape': {'width': 2.36659479,
   'height': 1.27,
   'depth': 1.18474138},
  'distance_from_user': 4.770507,
  'name': 'テーブル',
  'id': '313bc0b5-4320-4582-a28a-8748de4a9979'}}

In [157]:
from langchain.tools import tool
from typing import Annotated


@tool
def getFurnitureData(
    furniture_type: Annotated[str, "Type of furniture you want to refer to. 'TV' or 'TABLE' string data"],
    n_range: Annotated[float, "range: when you want to specify the limit range from the furniture to device. Default is 5"]
): 
    """You need this tool when you want to get devices around furniture you specified"""
    
    
    f_type = FurnitureType.TABLE.value if furniture_type.lower() == "table" else FurnitureType.TV.value
    
    print(f_type)
    sending = {
      "furnitureType": f_type,
        "range" : n_range
    }
    
    
    
    res = httpx.post(base_url + "/furniture/get", json=sending)
    data = res.json()
    data["devices"] = [{"id": i["id"], "name" : i["name"], "position_relative_to_furniture" : i["position"]} for i in data["devices"]]
    data["furniture_data"] = {
    "furnitureShape": data["furniture_data"]["furnitureShape"],
    "distance_from_user" : data["furniture_data"]["distance_from_user"],
    "name" : data["furniture_data"]["name"] ,
        "id":  data["furniture_data"]["id"]


    }


    
    return data


tool_map = {
    "getFurnitureData": getFurnitureData  # ← タイポです
}

In [158]:
from dataclasses import dataclass
from typing import Annotated, List
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

from langchain_core.messages import ToolMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
llm_with_tool = llm.bind_tools(tools=[getFurnitureData], strict=True)







@dataclass
class State:
    messages: Annotated[list, add_messages]


    
def tool_node(state):
    last_msg = state.messages[-1]
    this_tool = last_msg
    
    

def agent_node(state): 
    print("=========AGENT===========")
    
    agent_res = llm_with_tool.invoke(state.messages)
    print(agent_res)
    state.messages.append(agent_res) 
    return state




def tool_node(state):
    last_message = state.messages[-1]
    new_messages = []

    if len(last_message.tool_calls) == 0: 
        state.messages.append(
            ToolMessage(
                content="No tool call found, Please Try it again", 
                tool_call_id="0"
            )
        )
        return state

    for t in last_message.tool_calls:
        tool_name = t["name"]
        tool_args = t["args"]
        tool_id = t["id"]

        if tool_name not in tool_map:
            state.messages.append(
                ToolMessage(
                    content=f"Tool '{tool_name}' not recognized.", 
                    tool_call_id=tool_id
                )
            )
            continue

        tool_function = tool_map[tool_name]
        tool_output = tool_function.invoke(tool_args)

        print()
        print("TOOL OUTPUT:", tool_output)
        print()

        state.messages.append(
            ToolMessage(
                content=tool_output, 
                tool_call_id=tool_id
            )
        )

    return state



def router(state): 
    last_message = state.messages[-1]
  
    if last_message.tool_calls: 
        print("ツールを呼ぶ")
        return "tool_node"
    else: 
        print("ツールは呼ばない")
        return END
    
    
    


In [159]:
from langgraph.graph import START, END, StateGraph
from langchain_core.messages import SystemMessage, HumanMessage


graph=  StateGraph(State)

graph.add_node("agent_node", agent_node)
graph.add_node("tool_node",tool_node )

graph.add_edge(START, "agent_node")
graph.add_conditional_edges("agent_node", router)
graph.add_edge("tool_node", "agent_node")


runner = graph.compile()



In [162]:
messages = [
SystemMessage(content="You are visually supporting agent that helps user to find the devices. Please explain the position of the devices as concise and easily imagine and grasp the location of devices without watching"), 
    HumanMessage(content="机のまわりではどんな感じで電気が置かれている？机の上にあるのかないのか？机の高さを考えると、上にあるんじゃないの？")
]


init_state = State(messages = messages)



res = runner.invoke(init_state)
res

=========AGENT===========
content='' additional_kwargs={'tool_calls': [{'id': 'call_QjNyLXWneLFq9iC3p7QYXwp5', 'function': {'arguments': '{"furniture_type":"TABLE","n_range":5}', 'name': 'getFurnitureData'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 180, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86d0290411', 'finish_reason': 'tool_calls', 'logprobs': None} id='run-b4cf0355-cbcc-45d9-8832-31eebc1c0ef2-0' tool_calls=[{'name': 'getFurnitureData', 'args': {'furniture_type': 'TABLE', 'n_range': 5}, 'id': 'call_QjNyLXWneLFq9iC3p7QYXwp5', 'type': 'tool_call'}] usage_metadata={'input_tokens': 180, 'output_tokens': 23, 'total_tokens': 203, 'input_token_details': {'audio': 0, 

{'messages': [SystemMessage(content='You are visually supporting agent that helps user to find the devices. Please explain the position of the devices as concise and easily imagine and grasp the location of devices without watching', additional_kwargs={}, response_metadata={}, id='3b69dc0a-03cb-4fb2-92f3-3a24214ac3af'),
  HumanMessage(content='机のまわりではどんな感じで電気が置かれている？机の上にあるのかないのか？机の高さを考えると、上にあるんじゃないの？', additional_kwargs={}, response_metadata={}, id='509b24ad-52bc-40d1-afdf-b678881492e7'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QjNyLXWneLFq9iC3p7QYXwp5', 'function': {'arguments': '{"furniture_type":"TABLE","n_range":5}', 'name': 'getFurnitureData'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 180, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'au

In [161]:
res["messages"][-1].content

'机の周りには、2つのフロアライトがあります。\n\n1. **フロアライト**: 机の右側、少し後ろに位置しています。\n2. **フロアライト (1)**: 机の左側、少し後ろにあります。\n\n机の上には電気は置かれていないようです。'

In [163]:
output = {}


output["status"]  = "success"
output["devices"] = r

In [4]:
prompt = """

 # **SpatialReasoningAgent**

---

## Task

You are the **"SpatialReasoningAgent"** in a smart home environment powered by an LLM.  
Your primary responsibilities are to:

1. **Precisely identify the most appropriate device(s)** based on the user's spatial command.  
2. **Logically group devices** when multiple targets or groupings are involved.  
3. **Clearly format the output** for the next agent to carry out the control actions.  
4. **Filter out any non-applicable or inactive devices**, ensuring only meaningful selections remain.

---

## Input Data

You will receive the following inputs:

- **devices**: A list of devices, each with spatial coordinates relative to the user (x = left/right, z = front/back, y = height).  
- **filter_type**: The tool or method used to pre-filter the devices. Possible values include `"sight"`, `"direction"`, `"object"`, and `"around_furniture"`.  
- **order**: The sorting order applied before selection (e.g., `"proximity"`, `"right"`, `"height"`).  
- **user_prompt**: The user's original spatial command.  
  - Example: `"Turn on the two lights on the right in the back row."`  
  - Example: `"Turn on all the lights around me."`

---

## Spatial Reasoning Rules

### Singular Spatial Requests

Interpret the user's command using the following spatial rules:

- **Right / Left**:  
  - Right → device with the largest **positive x**.  
  - Left → device with the largest **negative x**.

- **Front / Back**:  
  - Front → device with the **smallest z** (closer to user).  
  - Back → device with the **largest z** (farther from user).

- **Highest / Lowest**:  
  - Highest → largest **y**.  
  - Lowest → smallest **y**.

---

### Compound Spatial Requests (e.g., “Right Back”)

Handle compound directions by prioritizing both axes:

- Example:  
  - **Right back** → max positive x **and** max positive z.  
  - **Left front** → max negative x **and** min z.

Use a **composite score** to rank candidates when multiple axes are mentioned:

```python
score = normalized_x + normalized_z
```

Normalize all axes before applying the scoring if necessary.

---

### Group Spatial Requests

Support requests that involve quantities or spatial groups:

- **Explicit Quantity** (e.g., "two rightmost lights")  
  → Select the exact number based on the sorting axis (e.g., descending x for rightmost).

- **Area-based** (e.g., "all around me")  
  → Select all devices within a reasonable radius (e.g., within 2.0 meters) around the user's (0,0,0) position.

- **Row-based** (e.g., "back row")  
  → Cluster by z-coordinate using a small threshold (e.g., ±0.2m) to identify rows, then sort within that cluster.

---

### Multiple Group Requests

If the user prompt refers to **multiple distinct regions** (e.g., "left front and right back"), identify and return each group independently.

Apply the reasoning rules for each group separately and include clear `"group_id"` and `"reason"` for each.

---

### Furniture-Based Spatial Reasoning

When `filter_type = "around_furniture"`, device positions are relative to a **furniture item** (not the user).  
Interpret coordinates **from the user's viewpoint looking toward the furniture**.

- **z-axis (front/back)**:  
  - Devices with **negative z** are **in front of the furniture** (closer to the user).  
  - Devices with **positive z** are **behind the furniture** (farther from the user).

- **x-axis (left/right)** and **y-axis (height)** remain relative to the furniture’s center.

Use this logic **only** when `filter_type = "around_furniture"` is provided.

---

### Prioritize User Perspective

Always interpret spatial terms from the user’s point of view unless explicitly overridden.

- **x-axis**:  
  - Positive → user's right  
  - Negative → user's left  

- **z-axis**:  
  - Positive → user's front (farther away)  
  - Negative → closer to the user  

- **y-axis**:  
  - Vertical height (used for "highest", "lowest", etc.)

---

## Handling Empty Selections

If **no devices match** the user's spatial command:

- Return an empty `selected_groups` list.  
- Include a `"reason"` field explaining why no match was found.

This ensures the system can gracefully handle cases where no action is required.

**Example:**

```json
{
  "selected_groups": [],
  "reason": "No devices matched the spatial command: 'Turn on the lights to the left of the table'."
}
```

---

## Output Format

Return your selection in the following JSON format.

### Example Prompt:
`"Turn on the two rightmost lights in the back row and two leftmost lights in the front row."`

### Expected Output:

```json
{
  "selected_groups": [
    {
      "group_id": "group_1",
      "devices": [
        { "id": "device_3" },
        { "id": "device_5" }
      ],
      "reason": "These are the two rightmost lights in the back row, identified by maximum positive x and maximum z-values."
    },
    {
      "group_id": "group_2",
      "devices": [
        { "id": "device_1" },
        { "id": "device_2" }
      ],
      "reason": "These are the two leftmost lights in the front row, identified by maximum negative x and minimum z-values."
    }
  ]
}
```

"""





prompt += """


ユーザー： "テーブルの下の電気を選択してください”


devices: {"status":"success","devices":[{"id":"d0fb08a8-981b-4006-959b-6ec5d3683f3d","name":"フロアライト","position":{"x":0.725365639,"y":0.768305957,"z":-0.06326866},"distance_from_user":4.652061,"angle":14.6652651},{"id":"03028511-86df-45b7-9037-3fed031eb43a","name":"フロアライト (1)","position":{"x":-0.3553723,"y":0.7668657,"z":-0.02977848},"distance_from_user":4.74021673,"angle":-4.50571775}],"furniture_data":{"position":{"x":-0.4238522,"y":-0.667920232,"z":4.70446253},"furnitureShape":{"width":2.36659479,"height":1.27,"depth":1.18474138},"distance_from_user":4.770507,"id":"8790899b-6876-40ad-9482-2db941786f8b","name":"テーブル","FurnitureType":"TABLE"}}


どのデバイスを選択しますか？

"""





In [8]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-4o", verbose=True)
res = llm.invoke(prompt)

In [9]:
res

AIMessage(content='```json\n{\n  "selected_groups": [\n    {\n      "group_id": "group_1",\n      "devices": [],\n      "reason": "No devices match the spatial command \'under the table\' as both devices are positioned above the table height."\n    }\n  ]\n}\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 1555, 'total_tokens': 1615, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_432e014d75', 'finish_reason': 'stop', 'logprobs': None}, id='run-b3a18cad-32f4-4928-9ae1-cadd2b4696ea-0', usage_metadata={'input_tokens': 1555, 'output_tokens': 60, 'total_tokens': 1615, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})